In [3]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Working folder:", Path.cwd())

files = [
    Path("data/calendar.csv"),
    Path("data/sell_prices.csv"),
    Path("data/sales_train_evaluation.csv"),
]

for f in files:
    print("✅" if f.exists() else "❌", f)

try:
    import xgboost as xgb
    print("✅ XGBoost:", xgb.__version__)
except Exception as e:
    print("❌ XGBoost:", type(e).__name__, e)

Python: 3.13.9
Pandas: 2.3.3
NumPy: 2.3.5
Working folder: C:\Users\iliasso
❌ data\calendar.csv
❌ data\sell_prices.csv
❌ data\sales_train_evaluation.csv
✅ XGBoost: 3.4.1


In [2]:
from pathlib import Path

PROJECT = Path(r"C:\Users\iliasso\AI_Demand_Pricing_Engine")

print("Project exists:", PROJECT.exists())

for name in [
    "calendar.csv",
    "sell_prices.csv",
    "sales_train_evaluation.csv"
]:
    path = PROJECT / "data" / name
    print("✅" if path.exists() else "❌", path)

Project exists: True
✅ C:\Users\iliasso\AI_Demand_Pricing_Engine\data\calendar.csv
✅ C:\Users\iliasso\AI_Demand_Pricing_Engine\data\sell_prices.csv
✅ C:\Users\iliasso\AI_Demand_Pricing_Engine\data\sales_train_evaluation.csv


In [5]:
calendar = pd.read_csv(PROJECT / "data" / "calendar.csv")
sell_prices = pd.read_csv(PROJECT / "data" / "sell_prices.csv")
sales = pd.read_csv(PROJECT / "data" / "sales_train_evaluation.csv")

print("calendar:", calendar.shape)
print("sell_prices:", sell_prices.shape)
print("sales:", sales.shape)

calendar: (1969, 14)
sell_prices: (6841121, 4)
sales: (30490, 1947)


In [6]:
# Keep the same store scope as the original project
sales_cal = sales.loc[sales["store_id"].eq("CA_1")].copy()

id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# Safety checks BEFORE reshaping
day_cols = [c for c in sales_cal.columns if c not in id_cols]

assert sales_cal.shape == (3049, 1947)
assert len(day_cols) == 1941
assert all(c.startswith("d_") for c in day_cols)

sales_long = sales_cal.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="demand"
)

# Safety checks AFTER reshaping
assert sales_long.shape == (5918109, 8)
assert sales_long["d"].str.startswith("d_").all()

print("✅ CA_1 rows:", sales_cal.shape)
print("✅ sales_long:", sales_long.shape)
print()
print(sales_long[["id", "item_id", "store_id", "d", "demand"]].head())

✅ CA_1 rows: (3049, 1947)
✅ sales_long: (5918109, 8)

                              id        item_id store_id    d  demand
0  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001     CA_1  d_1       0
1  HOBBIES_1_002_CA_1_evaluation  HOBBIES_1_002     CA_1  d_1       0
2  HOBBIES_1_003_CA_1_evaluation  HOBBIES_1_003     CA_1  d_1       0
3  HOBBIES_1_004_CA_1_evaluation  HOBBIES_1_004     CA_1  d_1       0
4  HOBBIES_1_005_CA_1_evaluation  HOBBIES_1_005     CA_1  d_1       0


In [7]:
dataset = sales_long.merge(
    calendar,
    on="d",
    how="left"
)

dataset = dataset.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# Validation
assert dataset.shape == (5918109, 22)
assert dataset["date"].isna().sum() == 0
assert dataset["sell_price"].isna().sum() == 1129842

print("✅ dataset shape:", dataset.shape)
print("✅ missing dates:", dataset["date"].isna().sum())
print("✅ missing prices:", dataset["sell_price"].isna().sum())

✅ dataset shape: (5918109, 22)
✅ missing dates: 0
✅ missing prices: 1129842


In [8]:
# ============================================================
# MODEL TABLE + TIME FEATURES + SAFE CHRONOLOGICAL ORDER
# ============================================================

model_data = dataset.dropna(subset=["sell_price"]).copy()

# Convert date correctly
model_data["date"] = pd.to_datetime(model_data["date"])

# Calendar features used by the original model
model_data["day_of_week"] = model_data["date"].dt.dayofweek
model_data["month"] = model_data["date"].dt.month
model_data["year"] = model_data["date"].dt.year

# Critical before creating lag / rolling features
model_data = (
    model_data
    .sort_values(["item_id", "date"])
    .reset_index(drop=True)
)

# Safety checks
assert len(model_data) == 4_788_267
assert model_data["sell_price"].isna().sum() == 0

bad_order = (
    model_data.groupby("item_id")["date"]
    .diff()
    .dropna()
    .lt(pd.Timedelta(0))
    .any()
)

assert bad_order == False

print("✅ model_data:", model_data.shape)
print("✅ missing prices:", model_data["sell_price"].isna().sum())
print("✅ chronological order verified")
print("✅ date range:", model_data["date"].min(), "→", model_data["date"].max())

✅ model_data: (4788267, 23)
✅ missing prices: 0
✅ chronological order verified
✅ date range: 2011-01-29 00:00:00 → 2016-05-22 00:00:00


In [9]:
model_data["demand_lag_1"] = (
    model_data.groupby("item_id")["demand"].shift(1)
)

model_data["demand_lag_7"] = (
    model_data.groupby("item_id")["demand"].shift(7)
)

model_data["rolling_mean_7"] = (
    model_data.groupby("item_id")["demand"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

model_data["rolling_mean_28"] = (
    model_data.groupby("item_id")["demand"]
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

print("✅ feature engineering complete")
print("✅ model_data shape:", model_data.shape)

✅ feature engineering complete
✅ model_data shape: (4788267, 27)


In [10]:
# ============================================================
# FINAL FEATURE TABLE + 28-DAY CHRONOLOGICAL HOLDOUT
# ============================================================

features = [
    "sell_price",
    "day_of_week",
    "month",
    "year",
    "demand_lag_1",
    "demand_lag_7",
    "rolling_mean_7",
    "rolling_mean_28"
]

# Remove rows where lag/rolling features are not yet available
ml_data = model_data.dropna(
    subset=features + ["demand"]
).copy()

# Final 28 days = untouched future holdout
cutoff_date = ml_data["date"].max() - pd.Timedelta(days=28)

train = ml_data.loc[ml_data["date"] <= cutoff_date].copy()
test = ml_data.loc[ml_data["date"] > cutoff_date].copy()

X_train = train[features]
y_train = train["demand"]

X_test = test[features]
y_test = test["demand"]

# Safety checks
assert X_train.shape == (4_617_523, 8)
assert X_test.shape == (85_372, 8)

assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

assert train["date"].max() < test["date"].min()

print("✅ ml_data:", ml_data.shape)
print("✅ cutoff:", cutoff_date)
print("✅ train dates:", train["date"].min(), "→", train["date"].max())
print("✅ test dates:", test["date"].min(), "→", test["date"].max())
print()
print("✅ X_train:", X_train.shape)
print("✅ y_train:", y_train.shape)
print("✅ X_test :", X_test.shape)
print("✅ y_test :", y_test.shape)
print("✅ no missing feature values")

✅ ml_data: (4702895, 27)
✅ cutoff: 2016-04-24 00:00:00
✅ train dates: 2011-02-26 00:00:00 → 2016-04-24 00:00:00
✅ test dates: 2016-04-25 00:00:00 → 2016-05-22 00:00:00

✅ X_train: (4617523, 8)
✅ y_train: (4617523,)
✅ X_test : (85372, 8)
✅ y_test : (85372,)
✅ no missing feature values


In [11]:
# ============================================================
# MODEL BENCHMARK
# Same training sample + same chronological future holdout
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


RANDOM_STATE = 42
N_COMPARE = 250_000

# ------------------------------------------------------------
# 1. Fixed training sample
# ------------------------------------------------------------

sample_idx = X_train.sample(
    n=N_COMPARE,
    random_state=RANDOM_STATE
).index

X_train_cmp = X_train.loc[sample_idx].copy()
y_train_cmp = y_train.loc[sample_idx].copy()

print("✅ Comparison train:", X_train_cmp.shape)
print("✅ Future holdout  :", X_test.shape)
print()


# ------------------------------------------------------------
# 2. Common evaluator
# ------------------------------------------------------------

results = []

def evaluate_model(name, model):
    print(f"Training {name} ...")
    
    start = time.perf_counter()

    model.fit(X_train_cmp, y_train_cmp)
    pred = model.predict(X_test)

    elapsed = time.perf_counter() - start

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_sec": elapsed
    })

    print(
        f"✅ {name} | "
        f"MAE={mae:.4f} | "
        f"RMSE={rmse:.4f} | "
        f"R²={r2:.4f} | "
        f"{elapsed:.1f}s"
    )
    print()


# ------------------------------------------------------------
# 3. Lag-1 baseline
# ------------------------------------------------------------

baseline_pred = X_test["demand_lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

results.append({
    "Model": "Lag-1 Baseline",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "R2": baseline_r2,
    "Training_Time_sec": 0.0
})

print(
    f"✅ Lag-1 Baseline | "
    f"MAE={baseline_mae:.4f} | "
    f"RMSE={baseline_rmse:.4f} | "
    f"R²={baseline_r2:.4f}"
)
print()


# ------------------------------------------------------------
# 4. Linear Regression
# ------------------------------------------------------------

evaluate_model(
    "Linear Regression",
    LinearRegression()
)


# ------------------------------------------------------------
# 5. Random Forest
# ------------------------------------------------------------

evaluate_model(
    "Random Forest",
    RandomForestRegressor(
        n_estimators=60,
        max_depth=14,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
)


# ------------------------------------------------------------
# 6. XGBoost
# ------------------------------------------------------------

evaluate_model(
    "XGBoost",
    XGBRegressor(
        n_estimators=250,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
)


# ------------------------------------------------------------
# 7. HistGradientBoosting — current configuration
# ------------------------------------------------------------

evaluate_model(
    "HistGradientBoosting",
    HistGradientBoostingRegressor(
        max_iter=100,
        learning_rate=0.1,
        max_depth=8,
        random_state=RANDOM_STATE
    )
)


# ------------------------------------------------------------
# 8. Final comparison
# ------------------------------------------------------------

comparison_df = (
    pd.DataFrame(results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

display(
    comparison_df.style.format({
        "MAE": "{:.4f}",
        "RMSE": "{:.4f}",
        "R2": "{:.4f}",
        "Training_Time_sec": "{:.1f}"
    })
)

print()
print("🏆 Best MAE:", comparison_df.iloc[0]["Model"])

✅ Comparison train: (250000, 8)
✅ Future holdout  : (85372, 8)

✅ Lag-1 Baseline | MAE=1.3477 | RMSE=2.7611 | R²=0.4108

Training Linear Regression ...
✅ Linear Regression | MAE=1.1334 | RMSE=2.1349 | R²=0.6477 | 0.4s

Training Random Forest ...
✅ Random Forest | MAE=1.0962 | RMSE=2.0483 | R²=0.6757 | 56.9s

Training XGBoost ...
✅ XGBoost | MAE=1.1029 | RMSE=2.0895 | R²=0.6626 | 18.4s

Training HistGradientBoosting ...
✅ HistGradientBoosting | MAE=1.1002 | RMSE=2.0555 | R²=0.6734 | 9.5s

FINAL MODEL COMPARISON


Matplotlib is building the font cache; this may take a moment.


,Model,MAE,RMSE,R2,Training_Time_sec
0,Random Forest,1.0962,2.0483,0.6757,56.9
1,HistGradientBoosting,1.1002,2.0555,0.6734,9.5
2,XGBoost,1.1029,2.0895,0.6626,18.4
3,Linear Regression,1.1334,2.1349,0.6477,0.4
4,Lag-1 Baseline,1.3477,2.7611,0.4108,0.0



🏆 Best MAE: Random Forest


In [12]:
comparison_path = PROJECT / "data" / "model_comparison_results.csv"

comparison_df.to_csv(comparison_path, index=False)

print("✅ Saved:", comparison_path)
print(comparison_df)

✅ Saved: C:\Users\iliasso\AI_Demand_Pricing_Engine\data\model_comparison_results.csv
                  Model       MAE      RMSE        R2  Training_Time_sec
0         Random Forest  1.096168  2.048321  0.675727          56.855441
1  HistGradientBoosting  1.100167  2.055530  0.673440           9.539490
2               XGBoost  1.102858  2.089465  0.662569          18.410270
3     Linear Regression  1.133447  2.134902  0.647734           0.440403
4        Lag-1 Baseline  1.347702  2.761078  0.410787           0.000000


In [13]:
# ============================================================
# RANDOM FOREST HYPERPARAMETER TUNING
# Chronological validation INSIDE the training period
# Final test set remains untouched
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit


RANDOM_STATE = 42
TUNING_SAMPLE = 150_000


# ------------------------------------------------------------
# 1. Create an internal 28-day validation window
#    using ONLY the original training period
# ------------------------------------------------------------

tune_cutoff = train["date"].max() - pd.Timedelta(days=28)

tune_history = train.loc[
    train["date"] <= tune_cutoff
].copy()

tune_validation = train.loc[
    train["date"] > tune_cutoff
].copy()


# Fixed historical sample for controlled compute
tune_sample = tune_history.sample(
    n=min(TUNING_SAMPLE, len(tune_history)),
    random_state=RANDOM_STATE
)

X_tune_train = tune_sample[features]
y_tune_train = tune_sample["demand"]

X_tune_val = tune_validation[features]
y_tune_val = tune_validation["demand"]


print("✅ Tuning train:", X_tune_train.shape)
print("✅ Validation  :", X_tune_val.shape)
print(
    "✅ Validation dates:",
    tune_validation["date"].min(),
    "→",
    tune_validation["date"].max()
)
print("✅ Final test set NOT used:", X_test.shape)
print()


# ------------------------------------------------------------
# 2. Build one explicit chronological CV split
# ------------------------------------------------------------

X_search = pd.concat(
    [X_tune_train, X_tune_val],
    ignore_index=True
)

y_search = pd.concat(
    [y_tune_train, y_tune_val],
    ignore_index=True
)

# -1 = training rows
#  0 = validation rows
test_fold = np.concatenate([
    np.full(len(X_tune_train), -1),
    np.zeros(len(X_tune_val), dtype=int)
])

time_split = PredefinedSplit(test_fold)


# ------------------------------------------------------------
# 3. Controlled search space
# ------------------------------------------------------------

param_distributions = {
    "n_estimators": [60, 80, 100],
    "max_depth": [10, 14, 18],
    "min_samples_leaf": [2, 5, 10],
    "max_features": [0.7, 1.0]
}


rf_tuning_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)


search = RandomizedSearchCV(
    estimator=rf_tuning_model,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="neg_mean_absolute_error",
    cv=time_split,
    random_state=RANDOM_STATE,

    # Avoid nested parallelism / unnecessary PC overload
    n_jobs=1,

    refit=True,
    verbose=2,
    return_train_score=False
)


# ------------------------------------------------------------
# 4. Run tuning
# ------------------------------------------------------------

search.fit(X_search, y_search)


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

tuning_results = pd.DataFrame(search.cv_results_)[
    [
        "params",
        "mean_test_score",
        "mean_fit_time"
    ]
].copy()

tuning_results["Validation_MAE"] = (
    -tuning_results["mean_test_score"]
)

tuning_results = (
    tuning_results
    .sort_values("Validation_MAE")
    .reset_index(drop=True)
)


print()
print("=" * 80)
print("RANDOM FOREST TUNING RESULTS")
print("=" * 80)

display(
    tuning_results[
        ["params", "Validation_MAE", "mean_fit_time"]
    ].style.format({
        "Validation_MAE": "{:.4f}",
        "mean_fit_time": "{:.1f}"
    })
)

print()
print("🏆 Best parameters:")
print(search.best_params_)

print()
print(
    "🏆 Best validation MAE:",
    round(-search.best_score_, 4)
)

print()
print("✅ Final chronological test set was NOT used for tuning.")

✅ Tuning train: (150000, 8)
✅ Validation  : (85372, 8)
✅ Validation dates: 2016-03-28 00:00:00 → 2016-04-24 00:00:00
✅ Final test set NOT used: (85372, 8)

Fitting 1 folds for each of 6 candidates, totalling 6 fits
[CV] END max_depth=14, max_features=0.7, min_samples_leaf=2, n_estimators=80; total time=  21.5s
[CV] END max_depth=18, max_features=1.0, min_samples_leaf=5, n_estimators=80; total time=  38.3s
[CV] END max_depth=18, max_features=1.0, min_samples_leaf=5, n_estimators=60; total time=  23.3s
[CV] END max_depth=10, max_features=1.0, min_samples_leaf=5, n_estimators=60; total time=  22.4s
[CV] END max_depth=18, max_features=0.7, min_samples_leaf=10, n_estimators=100; total time=  27.8s
[CV] END max_depth=10, max_features=0.7, min_samples_leaf=5, n_estimators=100; total time=  19.8s

RANDOM FOREST TUNING RESULTS


,params,Validation_MAE,mean_fit_time
0,"{'n_estimators': 100, 'min_samples_leaf': 5, 'max_features': 0.7, 'max_depth': 10}",1.0489,19.4
1,"{'n_estimators': 60, 'min_samples_leaf': 5, 'max_features': 1.0, 'max_depth': 10}",1.0501,22.2
2,"{'n_estimators': 100, 'min_samples_leaf': 10, 'max_features': 0.7, 'max_depth': 18}",1.0562,26.9
3,"{'n_estimators': 80, 'min_samples_leaf': 2, 'max_features': 0.7, 'max_depth': 14}",1.0593,20.8
4,"{'n_estimators': 80, 'min_samples_leaf': 5, 'max_features': 1.0, 'max_depth': 18}",1.0664,37.7
5,"{'n_estimators': 60, 'min_samples_leaf': 5, 'max_features': 1.0, 'max_depth': 18}",1.0669,22.6



🏆 Best parameters:
{'n_estimators': 100, 'min_samples_leaf': 5, 'max_features': 0.7, 'max_depth': 10}

🏆 Best validation MAE: 1.0489

✅ Final chronological test set was NOT used for tuning.


In [14]:
# ============================================================
# FINAL EVALUATION — TUNED RANDOM FOREST
# Same 250k benchmark training sample
# Untouched chronological final test
# ============================================================

import time
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


tuned_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    max_features=0.7,
    n_jobs=-1,
    random_state=42
)

print("Training tuned Random Forest...")

start = time.perf_counter()

tuned_rf.fit(X_train_cmp, y_train_cmp)

tuned_pred = tuned_rf.predict(X_test)

elapsed = time.perf_counter() - start

tuned_mae = mean_absolute_error(y_test, tuned_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_pred))
tuned_r2 = r2_score(y_test, tuned_pred)

print()
print("✅ FINAL TUNED RANDOM FOREST")
print(f"MAE  : {tuned_mae:.4f}")
print(f"RMSE : {tuned_rmse:.4f}")
print(f"R²   : {tuned_r2:.4f}")
print(f"Time : {elapsed:.1f}s")

print()
print("Previous untuned RF benchmark:")
print("MAE  : 1.0962")
print("RMSE : 2.0483")
print("R²   : 0.6757")

Training tuned Random Forest...

✅ FINAL TUNED RANDOM FOREST
MAE  : 1.0927
RMSE : 2.0388
R²   : 0.6787
Time : 38.3s

Previous untuned RF benchmark:
MAE  : 1.0962
RMSE : 2.0483
R²   : 0.6757


In [15]:
tuning_summary = pd.DataFrame([{
    "Model": "Random Forest",
    "n_estimators": 100,
    "max_depth": 10,
    "min_samples_leaf": 5,
    "max_features": 0.7,
    "Validation_MAE": 1.0489,
    "Final_Test_MAE": tuned_mae,
    "Final_Test_RMSE": tuned_rmse,
    "Final_Test_R2": tuned_r2,
    "Training_Time_sec": elapsed
}])

tuning_path = PROJECT / "data" / "random_forest_tuning_results.csv"

tuning_summary.to_csv(tuning_path, index=False)

print("✅ Saved:", tuning_path)
display(tuning_summary)

✅ Saved: C:\Users\iliasso\AI_Demand_Pricing_Engine\data\random_forest_tuning_results.csv


,Model,n_estimators,max_depth,min_samples_leaf,max_features,Validation_MAE,Final_Test_MAE,Final_Test_RMSE,Final_Test_R2,Training_Time_sec
0,Random Forest,100,10,5,0.7,1.0489,1.092742,2.038825,0.678727,38.264168
